# 🎙️ Kamba ASR — Whisper Small

**Model:** `openai/whisper-small`  
**Architecture:** Encoder-decoder Seq2Seq  
**Parameters:** ~244M  
**Dataset:** `DDD-Kenya/Kamba-ASR-Data-Subset-484H`  
**Training subset:** 10 hours  
**Validation:** 500 clips  
**Test:** 500 clips  
**Language token:** `sw` (Swahili — closest supported language in the Bash configuration)  
**Task:** `transcribe`  
**GPU:** T4 recommended

This notebook is the **Whisper Small portion of the supplied Bash script converted into a standalone Google Colab notebook**.


## 1. Runtime setup

In Colab select:

**Runtime → Change runtime type → T4 GPU**

Then run the cells from top to bottom.


In [ ]:
# GPU check
import subprocess, sys, os, platform

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU detected. In Google Colab select "
        "Runtime → Change runtime type → T4 GPU."
    )

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Install the packages used by the training pipeline
%pip install -q transformers==4.44.2 datasets==2.20.0 accelerate==0.33.0 jiwer evaluate


## 2. Mount Google Drive and create experiment directories

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, re, json, glob, logging, warnings, random
import numpy as np

DRIVE_ROOT = "/content/drive/MyDrive/KambaASR"
MODEL_KEY = "whisper-small"
MODEL_ROOT = f"{DRIVE_ROOT}/{MODEL_KEY}"

RAW_CACHE_DIR = f"{DRIVE_ROOT}/data_cache"
VOCAB_DIR = f"{DRIVE_ROOT}/ctc_tokenizer"

MODEL_DIR = f"{MODEL_ROOT}/best_model"
CKPT_DIR = f"{MODEL_ROOT}/checkpoints"
PRED_DIR = f"{MODEL_ROOT}/predictions"
OUTPUT_DIR = f"{MODEL_ROOT}/outputs"

for d in [RAW_CACHE_DIR, MODEL_DIR, CKPT_DIR, PRED_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Model root:", MODEL_ROOT)


## 3. Configuration

In [ ]:
from datasets import load_dataset, Dataset, Audio, load_from_disk
from transformers import set_seed

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATASET_ID = "DDD-Kenya/Kamba-ASR-Data-Subset-484H"
TRANSCRIPT_COL = "transcripts"
SAMPLE_RATE = 16_000

MAX_TRAIN_HOURS = 10.0
VAL_CLIPS = 500
TEST_CLIPS = 500

BATCH_SIZE = 4
GRAD_ACCUM = 8
NUM_EPOCHS = 10
LEARNING_RATE = 1e-5

WHISPER_MODEL_ID = "openai/whisper-small"
WHISPER_LANGUAGE = "sw"
WHISPER_TASK = "transcribe"

print("Dataset:", DATASET_ID)
print("Model:", WHISPER_MODEL_ID)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM)
print("Epochs:", NUM_EPOCHS)


## 4. Kamba text normalization

In [ ]:
_KAMBA = frozenset("abcdefghijklmnoprstuvwyĩũ")
_ALLOWED = _KAMBA | frozenset("0123456789") | frozenset("' ")
_APOS = str.maketrans({"’": "'", "‘": "'", "ʼ": "'", "`": "'"})
_PUNCT = re.compile(r'[.,;:?!\"()\[\]{}\-_/\\@#$%\^&\*+=|<>~`©]')

def normalise_text(text: str) -> str:
    if not text:
        return ""
    text = text.lower().translate(_APOS)
    text = _PUNCT.sub(" ", text)
    text = "".join(c for c in text if c in _ALLOWED)
    return re.sub(r" +", " ", text).strip()

def clean_label(text: str) -> str:
    return normalise_text(text)

assert normalise_text("Ng'ombe") == "ng'ombe"
assert "ĩ" in normalise_text("nĩ")
assert "ũ" in normalise_text("ũla")

print("Kamba normalizer ready.")


## 5. Stream and cache the Kamba dataset

In [ ]:
def stream_and_cache():
    max_sec = MAX_TRAIN_HOURS * 3600

    stream = load_dataset(
        DATASET_ID,
        split="train",
        streaming=True,
        trust_remote_code=True,
    )

    train_rows, val_rows, test_rows = [], [], []
    train_sec = 0.0

    for i, ex in enumerate(stream):
        if i % 500 == 0:
            logger.info(
                f"Scanned {i:,} | train={train_sec/3600:.2f}h "
                f"val={len(val_rows)} test={len(test_rows)}"
            )

        if len(test_rows) < TEST_CLIPS:
            test_rows.append(ex)
        elif len(val_rows) < VAL_CLIPS:
            val_rows.append(ex)
        elif train_sec < max_sec:
            try:
                sr = ex["audio"]["sampling_rate"] or SAMPLE_RATE
                dur = len(ex["audio"]["array"]) / sr
            except Exception:
                dur = 5.0

            train_rows.append(ex)
            train_sec += max(dur, 0.1)
        else:
            break

    os.makedirs(f"{RAW_CACHE_DIR}/shards", exist_ok=True)

    for rows, name in [
        (train_rows, "train"),
        (val_rows, "val"),
        (test_rows, "test"),
    ]:
        ds = Dataset.from_list(rows).cast_column(
            "audio",
            Audio(sampling_rate=SAMPLE_RATE),
        )
        ds.save_to_disk(f"{RAW_CACHE_DIR}/shards/{name}")
        logger.info(f"Saved {name}: {len(ds):,} clips")

    meta = {
        "dataset": DATASET_ID,
        "train_clips": len(train_rows),
        "val_clips": len(val_rows),
        "test_clips": len(test_rows),
        "train_hours": round(train_sec / 3600, 3),
    }

    with open(f"{RAW_CACHE_DIR}/meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    return meta


cache_exists = all(
    os.path.exists(f"{RAW_CACHE_DIR}/shards/{s}")
    for s in ["train", "val", "test"]
)

if not cache_exists:
    print("No existing dataset cache found. Streaming from Hugging Face...")
    meta = stream_and_cache()
else:
    with open(f"{RAW_CACHE_DIR}/meta.json") as f:
        meta = json.load(f)
    print("Existing Drive dataset cache found.")

print(json.dumps(meta, indent=2))


## 6. Load cached train/validation/test splits

In [ ]:
def load_splits():
    train = load_from_disk(
        f"{RAW_CACHE_DIR}/shards/train"
    ).cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    val = load_from_disk(
        f"{RAW_CACHE_DIR}/shards/val"
    ).cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    test = load_from_disk(
        f"{RAW_CACHE_DIR}/shards/test"
    ).cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    return train, val, test

raw_train, raw_val, raw_test = load_splits()

print(f"Train: {len(raw_train):,}")
print(f"Val:   {len(raw_val):,}")
print(f"Test:  {len(raw_test):,}")


## 7. Whisper processor

In [ ]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    GenerationConfig,
)

print("Loading Whisper processor...")

processor = WhisperProcessor.from_pretrained(
    WHISPER_MODEL_ID,
    language=WHISPER_LANGUAGE,
    task=WHISPER_TASK,
)

processor.save_pretrained(f"{MODEL_ROOT}/processor")

print("Processor saved to:", f"{MODEL_ROOT}/processor")


## 8. Whisper preprocessing

In [ ]:
def preprocess_whisper(batch, processor, language="sw", task="transcribe"):
    """Extract Whisper log-mel features and tokenize normalized transcripts."""

    audio_arrays = [a["array"] for a in batch["audio"]]
    sr = batch["audio"][0]["sampling_rate"]

    features = processor.feature_extractor(
        audio_arrays,
        sampling_rate=sr,
        return_tensors="pt",
    )

    batch["input_features"] = features.input_features

    clean_texts = [
        normalise_text(t)
        for t in batch[TRANSCRIPT_COL]
    ]

    batch["labels"] = processor.tokenizer(
        clean_texts,
        return_tensors="pt",
        padding=False,
    ).input_ids

    return batch


def encode_whisper_dataset(ds, split_name):
    return ds.map(
        lambda batch: preprocess_whisper(
            batch,
            processor,
            language=WHISPER_LANGUAGE,
            task=WHISPER_TASK,
        ),
        batched=True,
        batch_size=8,
        remove_columns=ds.column_names,
        desc=f"Encoding {split_name}",
    )


print("Whisper preprocessing functions ready.")


## 9. Encode the datasets

In [ ]:
print("Encoding train...")
train_ds = encode_whisper_dataset(raw_train, "train")

print("Encoding validation...")
val_ds = encode_whisper_dataset(raw_val, "val")

print("Encoding test...")
test_ds = encode_whisper_dataset(raw_test, "test")

print("Encoding complete.")
print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))


## 10. Whisper data collator

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Whisper collator: pads input features and labels separately."""

    processor: Any

    def __call__(self, features):
        input_features = [
            {"input_features": f["input_features"]}
            for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        # Remove SOT/BOS token if present, matching the Bash implementation.
        if labels.shape[1] > 0 and (
            labels[:, 0] == self.processor.tokenizer.bos_token_id
        ).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

print("Whisper data collator ready.")


## 11. WER/CER metrics

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_whisper_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    preds = [
        normalise_text(s)
        for s in processor.tokenizer.batch_decode(
            pred_ids,
            skip_special_tokens=True,
        )
    ]

    refs = [
        normalise_text(s)
        for s in processor.tokenizer.batch_decode(
            label_ids,
            skip_special_tokens=True,
        )
    ]

    pairs = [
        (p, r)
        for p, r in zip(preds, refs)
        if r.strip()
    ]

    if not pairs:
        return {"wer": 1.0, "cer": 1.0}

    ps, rs = zip(*pairs)

    wer = wer_metric.compute(
        predictions=list(ps),
        references=list(rs),
    )

    cer = cer_metric.compute(
        predictions=list(ps),
        references=list(rs),
    )

    return {
        "wer": round(wer, 4),
        "cer": round(cer, 4),
    }

print("WER/CER metrics ready.")


## 12. Build Whisper Small

In [ ]:
print("Loading Whisper Small...")

model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_MODEL_ID
)

# Match the Bash script.
model.generation_config.language = WHISPER_LANGUAGE
model.generation_config.task = WHISPER_TASK
model.generation_config.forced_decoder_ids = None

total = sum(p.numel() for p in model.parameters())
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total/1e6:.1f}M")
print(f"Trainable parameters: {trainable/1e6:.1f}M")


## 13. Training configuration

In [ ]:
# Check for an existing checkpoint so training can resume.
ckpts = [
    d for d in os.listdir(CKPT_DIR)
    if d.startswith("checkpoint-")
]

resume_from = None

if ckpts:
    resume_from = os.path.join(
        CKPT_DIR,
        sorted(
            ckpts,
            key=lambda d: int(d.split("-")[-1])
        )[-1],
    )
    print("⏪ Resuming from:", resume_from)
else:
    print("No checkpoint found. Training from scratch.")

training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=(torch.cuda.is_available()),

    evaluation_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    logging_steps=25,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    gradient_checkpointing=True,

    predict_with_generate=True,
    generation_max_length=225,
    remove_unused_columns=False,
)

print("Training arguments ready.")


## 14. Create the Seq2SeqTrainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_whisper_metrics,
    tokenizer=processor.feature_extractor,
)

print("Trainer ready.")


## 15. Train Whisper Small

In [ ]:
print("=" * 70)
print("🚀 TRAINING WHISPER SMALL ON KAMBA")
print("=" * 70)

train_result = trainer.train(
    resume_from_checkpoint=resume_from
)

print("Training complete.")


## 16. Save the best model and processor

In [ ]:
trainer.save_model(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)

print("✅ Model saved:", MODEL_DIR)
print("✅ Processor saved:", MODEL_DIR)


## 17. Evaluate on validation set

In [ ]:
print("Evaluating validation set...")

val_metrics = trainer.evaluate(
    val_ds,
    metric_key_prefix="val",
)

print(json.dumps(val_metrics, indent=2))


## 18. Evaluate on test set

In [ ]:
print("Evaluating test set...")

test_metrics = trainer.evaluate(
    test_ds,
    metric_key_prefix="test",
)

print(json.dumps(test_metrics, indent=2))


## 19. Save Whisper Small metrics

In [ ]:
from datetime import datetime

results = {
    "model_key": MODEL_KEY,
    "model_id": WHISPER_MODEL_ID,
    "type": "seq2seq",
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),

    "train_hours": MAX_TRAIN_HOURS,
    "train_clips": len(train_ds),

    "epochs": NUM_EPOCHS,
    "lr": LEARNING_RATE,

    "train_loss": round(train_result.training_loss, 4),

    "val_wer": val_metrics.get("val_wer"),
    "val_cer": val_metrics.get("val_cer"),

    "test_wer": test_metrics.get("test_wer"),
    "test_cer": test_metrics.get("test_cer"),

    "val_score": round(
        (
            1
            - 0.5 * val_metrics.get("val_wer", 1)
            - 0.5 * val_metrics.get("val_cer", 1)
        ) * 100,
        2,
    ),
}

metrics_path = f"{MODEL_ROOT}/metrics.json"

with open(metrics_path, "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print("Metrics saved to:", metrics_path)


## 20. Final Whisper Small results

The Bash script uses WER and CER as the main evaluation metrics. The previously recorded Wav2Vec-BERT 2.0 result was:

- **Validation WER:** 28.66%
- **Validation CER:** 8.27%
- **Score:** 81.54%

Your Whisper result below can be compared against that baseline.


In [ ]:
print("=" * 70)
print("WHISPER SMALL — FINAL RESULTS")
print("=" * 70)

print(f"Validation WER: {results['val_wer'] * 100:.2f}%")
print(f"Validation CER: {results['val_cer'] * 100:.2f}%")
print(f"Test WER:       {results['test_wer'] * 100:.2f}%")
print(f"Test CER:       {results['test_cer'] * 100:.2f}%")
print(f"Validation score: {results['val_score']:.2f}")

print("\nPreviously recorded Wav2Vec-BERT 2.0:")
print("Validation WER: 28.66%")
print("Validation CER: 8.27%")
print("Score:          81.54")


## 21. Reload results later

If training has already completed, this cell lets you inspect the saved metrics without retraining.


In [ ]:
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        saved_results = json.load(f)

    print(json.dumps(saved_results, indent=2))
else:
    print("No metrics.json found yet.")
